# 19b. Agent Loop (Minimal)

**Tier:** Agent Engineering
**Estimated time:** 15 minutes
**Prerequisites:** 18, 19
**Priority:** 🟢 Nice-to-have — a condensed reference version of 19. *If skipped, revisit when:* when you need a copy-paste starting point.
**Source material:** @sairahul1 ; companion to notebook 19

## What You'll Learn
- The smallest agent loop that actually works — ~30 lines, one tool, copy-paste ready
- When to reach for this minimal version versus the fully-instrumented build in notebook 19

## Why This Matters
Notebook 19 printed every step so you could *understand* the loop. Once you understand it, you rarely want all that scaffolding — you want the loop, one or two tools, and nothing else. This is the version you actually paste into a project. **Use 19 to learn and debug; use 19b to ship.**


## Same loop, nothing extra

Notebook 19's loop and this one are the *same algorithm*. The difference is everything around it: 19 has a tool registry, trace printing, instrumentation, and a scratch directory; 19b has a tools dict and a `while` loop. Nothing here is new — it's notebook 19 with the teaching scaffolding removed.

The trade is explicitness vs. brevity:
- **Notebook 19** — you can see every message, every `tool_use_id`, every decision. Best when something's broken and you need to know *why*.
- **Notebook 19b** — you can read the whole agent in one screen. Best when it works and you just want it in your codebase.


In [ ]:
import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"   # production default: claude-opus-4-8

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("ready")
else:
    client = None
    print("No ANTHROPIC_API_KEY — the loop will skip its live call.")


## The whole agent

One tool, one schema, one loop. Read it top to bottom — there's no hidden machinery.


In [ ]:
# 1. One tool: a schema the model reads + the function we run.
def calculator(expression):
    return str(eval(expression, {"__builtins__": {}}))

TOOLS = {"calculator": calculator}
SCHEMAS = [{"name": "calculator", "description": "Evaluate an arithmetic expression.",
            "input_schema": {"type": "object",
                             "properties": {"expression": {"type": "string"}},
                             "required": ["expression"]}}]

# 2. The minimal loop.
def agent(goal, max_steps=6):
    messages = [{"role": "user", "content": goal}]
    for _ in range(max_steps):
        resp = client.messages.create(model=TEACH_MODEL, max_tokens=400,
                                       tools=SCHEMAS, messages=messages)
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")
        messages.append({"role": "assistant", "content": resp.content})
        messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": b.id, "content": TOOLS[b.name](**b.input)}
            for b in resp.content if b.type == "tool_use"
        ]})
    return "stopped: max_steps reached"

# 3. Run it.
if HAS_ANTHROPIC:
    print(agent("What is 1234 * 5678? Use the calculator, then state the result in a sentence."))
else:
    print("  [skipped: no ANTHROPIC_API_KEY]")


That's a complete, working agent in ~20 lines of logic. Everything in notebook 19 — multiple tools, budgets, tracing — is an *addition* to this core, not a change to it.


## Exercises


In [ ]:
# Exercise 1 (Warm-up): Swap the tool
# Task: Replace the calculator with a `reverse_string` tool and ask the agent to reverse a word.
#       Keep the loop completely unchanged.
# Hint: Only TOOLS and SCHEMAS change — that's the proof the loop is tool-agnostic.

# YOUR CODE HERE


In [ ]:
# Exercise 2 (Apply): Two tools in the minimal loop
# Task: Add a second tool to TOOLS and SCHEMAS (e.g. word_count). Confirm the same loop handles
#       two tools with no other edits.
# Hint: The dict-comprehension in the loop already iterates over ALL tool_use blocks, so multiple
#       tools per turn already work.

# YOUR CODE HERE


In [ ]:
# Exercise 3 (Extend): Decide which version you'd use
# Task: In a comment, describe one real situation where you'd reach for notebook 19's instrumented
#       version instead of this minimal one, and one where you'd prefer this one.
# Hint: Think about debugging a misbehaving agent (which gives you visibility?) vs. embedding a
#       known-good agent into a larger system (which is less to maintain?).

# YOUR CODE HERE


<details>
<summary>Show solutions</summary>

```python
# Exercise 1
def reverse_string(text): return text[::-1]
TOOLS = {"reverse_string": reverse_string}
SCHEMAS = [{"name": "reverse_string", "description": "Reverse a string.",
            "input_schema": {"type": "object", "properties": {"text": {"type": "string"}},
                             "required": ["text"]}}]
if HAS_ANTHROPIC: print(agent("Reverse the word 'robotics'."))

# Exercise 2
def word_count(text): return str(len(text.split()))
TOOLS["word_count"] = word_count
SCHEMAS.append({"name": "word_count", "description": "Count words in a string.",
                "input_schema": {"type": "object", "properties": {"text": {"type": "string"}},
                                 "required": ["text"]}})
if HAS_ANTHROPIC: print(agent("Reverse 'hello world' and also count its words."))

# Exercise 3
# Use notebook 19 (instrumented) when an agent is making wrong tool choices or looping and you
# need to SEE each message and stop_reason to diagnose it. Use 19b when the agent is proven and
# you're embedding it in a service where less code = less to maintain and review.
```
</details>


## Key Takeaways
- The minimal agent is ~20 lines: the same call → tool → result → repeat loop as notebook 19, with the scaffolding removed.
- The loop is tool-agnostic — adding or swapping tools touches only the TOOLS dict and SCHEMAS list.
- Reach for the instrumented version (19) to learn and debug; reach for the minimal version (19b) to ship.
- The dict-comprehension over `tool_use` blocks already handles multiple tools per turn for free.

## What's Next
Notebook **20 — Multi-Agent Patterns** stops hand-writing the loop and introduces **LangGraph**, which turns the loop into an explicit graph of nodes and edges — and makes coordinating *several* agents tractable.
